### ESPORTAZIONE IN FILE JSON per D3

Viene creato un dizionario contenente:
- regole, 
- contro regole, 
- esemplari, 
- statistiche del dataset, 
- valori dell'istanza, 
- feature importance SHAP
- D.t. pred
- bb pred

Il dizionario viene poi esportato in JSON

In [1]:
import pandas as pd
import numpy as np


from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt
import pickle

import os

ModuleNotFoundError: No module named 'lore'

In [ ]:
import xgboost as xgb
import dill #aneto
import SuperLore
import category_encoders
import re
import json
import os

#### Carico il Modello

In [ ]:
bb = xgb.XGBClassifier()
bb.load_model("../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_train.model")

In [ ]:
bb

#### Caricol x train, y train, x test, y test

In [2]:
X_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtrain')
Y_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytrain')
X_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtest')
Y_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytest')
data_desc=pd.read_pickle('../datasets/Dati-Banca-Lore/intesa_incassi_data_description.p')
path =('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_explanations_full.p')
explanations_shap = dill.load(open(path, 'rb'))

FileNotFoundError: [Errno 2] No such file or directory: '../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtrain'

### DATA DESCRIPTION

In [11]:
type(data_desc)
data_desc

{'numerical': {'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': {'mean': 0.4963,
   'std': 0.1677,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.4274,
   '2nd-quantile': 0.5059,
   '3rd-quantile': 0.5592},
  'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': {'mean': 0.4103,
   'std': 0.2236,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.246,
   '2nd-quantile': 0.3661,
   '3rd-quantile': 0.5307},
  'PRODV_LETTERE_DI_CREDITO_PON': {'mean': 0.5002,
   'std': 0.1472,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.453,
   '2nd-quantile': 0.453,
   '3rd-quantile': 0.453},
  'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': {'mean': 0.4061,
   'std': 0.2854,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.1745,
   '2nd-quantile': 0.3338,
   '3rd-quantile': 0.6341},
  'PN_SEPA_ENTRATA_TY_VAL': {'mean': 0.4587,
   'std': 0.2845,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.2157,
   '2nd-quantile': 0.4202,
   '3rd-quantile': 0.7006},
  'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': {'mean': 0.3707,
   's

#### Carico le spiegazioni di Lore

In [12]:
lore_exp_path=open('../datasets/Dati-Banca-Lore/lore_exp_INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xgb_cfs_binary_from_dts.p','rb')
objs = []
while 1:
    try:
        objs.append(pickle.load(lore_exp_path))
    except EOFError:
        break

In [13]:
for lore_p in objs[0].rule.premises:
    print(lore_p)

PN_SEPA_USCITA_TY_NUM = 1.00
SCADV_FLG_RATA_DIVISA_SEK = 1.00


### Carico SHAP

In [14]:
feature_names = X_test.columns
fi_shap={}
for j,f in enumerate(feature_names):
        fi_shap[f]= dict()
        fi_shap[f]['feature_importance'] = explanations_shap[0][j]
print(fi_shap)

{'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': {'feature_importance': 0.09466361575103292}, 'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': {'feature_importance': 0.08120621911794842}, 'PRODV_LETTERE_DI_CREDITO_PON': {'feature_importance': 0.2854290714347371}, 'SCADV_FLG_RATA_DIVISA_SEK': {'feature_importance': 0.0}, 'OWNER_PRODV_FOREX_PON': {'feature_importance': 0.3347166349758118}, 'OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON': {'feature_importance': 0.11593095222277043}, 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': {'feature_importance': 0.03719840600451789}, 'PN_LC_IMPORT_FLG_ONLY_TY': {'feature_importance': 0.0}, 'PN_SEPA_ENTRATA_TY_VAL': {'feature_importance': 0.1827153803166584}, 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': {'feature_importance': -0.02485413807482473}, 'OWNER_PRODV_GARANZIE_DOMESTICHE_PON': {'feature_importance': 0.010956627331615891}, 'PN_SEPA_USCITA_PRC_DLT_YEAR_VAL': {'feature_importance': -0.03174290419792669}, 'SCADV_FLG_RATA_DIVISA_AUD': {'feature_importance': 0.0}, 'OWNER_PRODV_AN

### CREO il dizionario

In [15]:
rule_list=[]
def parse_rule(lore_obj): #parso le rule
    for lore_r in lore_obj.rule.premises:
        rule_list.append(vars(lore_r))    
    return rule_list

counterrule_list=[]
def parse_counterrules(lore_obj): #parso le counterrule
    for cr in lore_obj.crules:
        for nested in cr.premises:
            counterrule_list.append(vars(nested))
    return counterrule_list

exemplars_list = []
def parse_exemplars(lore_obj): #parso gli exemplars
    ex=lore_obj.exemplars
    new_ex = re.sub(r'(\d\.\d+)', r'\1 ', ex) #separo numeri da istanze
    pattern = r'{(.*?)}'
    matches = re.findall(pattern, new_ex, re.DOTALL)
    for match in matches:
        ex_parse = {}
        list_to_dict=[]
        properties = match.split() #ogni stringa separata da uno spazio diventa un nuovo item della lista, ogni lista è un exemplar
        for prop in properties: 
            if prop != "=": #elimino =
                list_to_dict.append(prop) # creo un dizionario con chiave e valore
        for i in range(0, len(list_to_dict), 2):
            ex_parse[list_to_dict[i]] = float(list_to_dict[i + 1])
        exemplars_list.append(ex_parse)
    return exemplars_list

dt_dict={}
def parse_dt(lore_obj): #parso dt, qui biasogna implementare il parsing dell'oggetto tree, che per ora sto sostiutendo con una stringa
    dt_dict=lore_obj.dt.__dict__
    dt_dict.update(
    {'tree_':'to add'
    })
    return dt_dict


def inst_value (n):
    inst= X_train.iloc[n]
    inst_dict = inst.to_dict()
    return inst_dict



In [16]:
inst= X_train.iloc[0]
inst_dict= inst.to_dict()
print(inst_dict)

{'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': 0.1901915037018963, 'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': 0.17117117117117117, 'PRODV_LETTERE_DI_CREDITO_PON': 0.45295295295295296, 'SCADV_FLG_RATA_DIVISA_SEK': 0.0, 'OWNER_PRODV_FOREX_PON': 0.0, 'OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON': 0.0, 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': 0.8998171725444453, 'PN_LC_IMPORT_FLG_ONLY_TY': 0.0, 'PN_SEPA_ENTRATA_TY_VAL': 0.9697662572726942, 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': 0.3412673027139967, 'OWNER_PRODV_GARANZIE_DOMESTICHE_PON': 1.0, 'PN_SEPA_USCITA_PRC_DLT_YEAR_VAL': 0.5515515515515516, 'SCADV_FLG_RATA_DIVISA_AUD': 0.0, 'OWNER_PRODV_ANTICIPI_EXPORT_PON': 0.0, 'PN_SWIFT_ENTRATA_FLG_NEW_TRANS': 0.0, 'SCADV_FLG_RATA_DIVISA_CAD': 0.0, 'OWNER_PRODV_DCM_PON': 0.0, 'PCRIV_FT_20_DLT_ANNUO_UTILZZ_ACCRD_MEDIO': 0.14895620781666932, 'SCADV_FLG_RATA_DIVISA_JPY': 0.0, 'PN_SEPA_USCITA_TY_NUM': 0.9549537764095559, 'PRODV_FACTORING_PON': 0.4624624624624625, 'PRODV_SECURITIZATION_PON': 0.4984984984984985, 'BES

In [17]:
def parse_dict_first(lore_obj): #creo un dizionario dell'istanza, alcuni oggetti però vanno parsati meglio
    istance = lore_obj.__dict__
    return istance

def update_object(istance):
    istance.update(
    {'rule': rule_list,
     'crules':counterrule_list,
     'exemplars':exemplars_list,
     'dt':dt_dict,
     'distribution':data_desc,
     'inst':inst_dict,
     'fi_shap':fi_shap
    })
    return istance   


In [18]:
istance_0 = parse_dict_first(objs[0])
parse_rule(objs[0])
parse_counterrules(objs[0])
parse_exemplars(objs[0])
parse_dt(objs[0])
inst_value(0)

update_object(istance_0)
print(istance_0)

{'bb_pred': 0, 'dt_pred': 0, 'rule': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'SCADV_FLG_RATA_DIVISA_SEK', 'op': '=', 'thr': 1, 'is_continuous': False}], 'crules': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_FOREX_PON', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL', 'op': '=', 'thr': 1, 'is_continuous': False}], 'deltas': [[]], 'feature_importance': {'SCADV_FLG_RATA_DIVISA_SEK': 0.050431115237558576, 'PN_SEPA_USCITA_TY_NUM': 0.3623076093597704}, 'feature_importance_all': {'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': 0.0, 'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': 0.04223294247787269, 'PRODV_LETTERE_DI_CREDITO_PON': 0.0, 'SCADV_FLG_RATA_DIVISA_SEK': 0.050431115237558576, 'OWNER_PRODV_FOREX_PON': 0.24157065287061502, 'OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON': 0.

In [21]:
istance_1 = parse_dict_first(objs[1])
parse_rule(objs[1])
parse_counterrules(objs[1])
parse_exemplars(objs[1])
parse_dt(objs[1])
inst_value(1)

update_object(istance_1)
print(istance_1)


{'bb_pred': 0, 'dt_pred': 0, 'rule': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'SCADV_FLG_RATA_DIVISA_SEK', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}], 'crules': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_FOREX_PON', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCAD

In [22]:
istance_2 = parse_dict_first(objs[2])
parse_rule(objs[2])
parse_counterrules(objs[2])
parse_exemplars(objs[2])
parse_dt(objs[2])
inst_value(2)

update_object(istance_2)
print(istance_2)

{'bb_pred': 1, 'dt_pred': 1, 'rule': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'SCADV_FLG_RATA_DIVISA_SEK', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PCRIV_NAT_FT_03_LAST_YYYYMM_UTILZZ_ACCRDT_PERC', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PRODV_SECURITIZATION_PON', 'op': '=', 'thr': 1, 'is_continuous': False}], 'crules': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_FOREX_PON', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 0, 'is_continuous': False}, 

In [23]:
istance_3 = parse_dict_first(objs[3])
parse_rule(objs[3])
parse_counterrules(objs[3])
parse_exemplars(objs[3])
parse_dt(objs[3])

update_object(istance_3)
print(istance_3)

{'bb_pred': 0, 'dt_pred': 0, 'rule': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'SCADV_FLG_RATA_DIVISA_SEK', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PCRIV_NAT_FT_03_LAST_YYYYMM_UTILZZ_ACCRDT_PERC', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PRODV_SECURITIZATION_PON', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL', 'op': '=', 'thr': 1, 'is_continuous': False}], 'crules': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_FOREX_PON', 'op': '=', 'thr': 1, 'is_continuous': False},

In [24]:
istance_4 = parse_dict_first(objs[4])
parse_rule(objs[4])
parse_counterrules(objs[4])
parse_exemplars(objs[4])
parse_dt(objs[4])

update_object(istance_4)
print(istance_4)

{'bb_pred': 1, 'dt_pred': 1, 'rule': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'SCADV_FLG_RATA_DIVISA_SEK', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PCRIV_NAT_FT_03_LAST_YYYYMM_UTILZZ_ACCRDT_PERC', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PRODV_SECURITIZATION_PON', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL', 'op': '=', 'thr': 1, 'is_continuous'

In [25]:
istance_5 = parse_dict_first(objs[5])
parse_rule(objs[5])
parse_counterrules(objs[5])
parse_exemplars(objs[5])
parse_dt(objs[5])

update_object(istance_5)
print(istance_5)

{'bb_pred': 1, 'dt_pred': 1, 'rule': [{'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'SCADV_FLG_RATA_DIVISA_SEK', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SEPA_USCITA_TY_NUM', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PCRIV_NAT_FT_03_LAST_YYYYMM_UTILZZ_ACCRDT_PERC', 'op': '=', 'thr': 0, 'is_continuous': False}, {'att': 'PRODV_SECURITIZATION_PON', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'OWNER_PRODV_ANTICIPI_EXPORT_PON', 'op': '=', 'thr': 1, 'is_continuous': False}, {'att': 'PN_SWIFT_USCITA_PRC_DLT_YEAR_VAL', 'op': '=', 'thr': 1, 'is_continuous'

#### ESPORTO IL JSON

In [19]:
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

In [30]:
filename = "../JSON_istance/istance_5.json"
with open(filename, "w") as file:
    json.dump(istance_5, file, cls=NpEncoder)